# CVE → CWE Embedding Fine-Tuning

Side-by-side comparison of two training recipes, both scored by the **same two-stage hierarchical evaluator**:

> **Category → BFS top-k**

## Training Recipes

### A_hier_full
- CVE → CWE with **sibling + other-category hard negatives**
- CVE → category
- CWE-level sibling geometry

### B_children_only
- Plain **(CVE, own CWE)** pairs
- In-batch negatives only
- **No** mined / random / sibling negatives at all

## Controlled Variables

Both recipes use the same:

- Split
- Base model + LoRA initialization
- Number of epochs
- Learning rate
- Batch size
- Random seed

Therefore, differences in the final table come from the **training data alone**.

In [2]:
!pip install -U "transformers>=5.2,<=5.5.0" "trl<=0.24.0" peft sentence-transformers unsloth unsloth_zoo

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.29.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached torchvision-0.29.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 131.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/

In [3]:
from unsloth import FastSentenceTransformer


def load_model():
    """Fresh base model + fresh LoRA adapters (identical init on every call)."""
    m = FastSentenceTransformer.from_pretrained(
        model_name = "unsloth/all-MiniLM-L6-v2",
        max_seq_length = 512,
        full_finetuning = False,
    )
    return FastSentenceTransformer.get_peft_model(
        m,
        r = 64,
        target_modules = ['value', 'key', 'dense', 'query'],
        lora_alpha = 128,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = False,
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
        task_type = "FEATURE_EXTRACTION",
    )

/usr/local/lib/python3.11/dist-packages/unsloth/_gpu_init.py:110: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Could not load this library: /usr/local/lib/python3.11/dist-packages/torchaudio/lib/libtorchaudio.so
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Load data

In [4]:
import ast, random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv("hf://datasets/regularpooria/CVE_CWE_Software_Mapping_Dataset/Global_Dataset.csv")
df = df.dropna(
    subset=["CVE-ID", "CVE-DESCRIPTION-NORMALIZED", "CWE-ID", "CWE-DESCRIPTION"]
).reset_index(drop=True)
print(f"Loaded {len(df):,} rows, {df['CVE-ID'].nunique():,} CVEs, {df['CWE-ID'].nunique():,} CWEs")


Loaded 60,897 rows, 60,092 CVEs, 146 CWEs


### Taxonomy: nodes, parents/children, BFS helpers


In [5]:
def parse_parents(parents_raw):
    """PARENTS is a stringified list of (cwe_id, description) tuples."""
    if parents_raw is None or (isinstance(parents_raw, float) and pd.isna(parents_raw)):
        return []
    if isinstance(parents_raw, list):
        return parents_raw
    try:
        parsed = ast.literal_eval(parents_raw)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return []
class Taxonomy:
    """
    Nodes   = every CWE seen as a row OR as a parent.
    Category = any node that has at least one child (a "parent CWE").
    A node can have several parents, so this is a DAG; all walks are cycle-safe.
    """

    def __init__(self, df):
        own_text, parent_text = {}, {}
        parents = defaultdict(set)
        for cwe_id, desc, raw in zip(df["CWE-ID"], df["CWE-DESCRIPTION"], df["PARENTS"]):
            own_text[cwe_id] = desc
            for pid, pdesc in parse_parents(raw):
                parents[cwe_id].add(pid)
                parent_text[pid] = pdesc
        self.text = {**parent_text, **own_text}          # a CWE's own row wins
        self.parents = dict(parents)
        children = defaultdict(set)
        for c, ps in self.parents.items():
            for p in ps:
                children[p].add(c)
        self.children = dict(children)
        self.ids = sorted(self.text)
        self.idx = {n: i for i, n in enumerate(self.ids)}
        self.categories = sorted(self.children)

    def _walk(self, start, edges, max_depth):
        seen, frontier, out = {start}, [start], []
        for _ in range(max_depth):                       # breadth-first
            nxt = []
            for n in frontier:
                for m in edges.get(n, ()):
                    if m not in seen:
                        seen.add(m); out.append(m); nxt.append(m)
            frontier = nxt
            if not frontier:
                break
        return out

    def descendants(self, n, max_depth=2):
        return self._walk(n, self.children, max_depth)

    def ancestors(self, n, max_depth=3):
        return self._walk(n, self.parents, max_depth)

    def related(self, n):
        """n itself plus everything above/below it: never a valid negative."""
        return {n, *self.ancestors(n, 3), *self.descendants(n, 3)}

    def siblings(self, n):
        """Other children of any of n's parents (excluding ancestors/descendants)."""
        s = set()
        for p in self.parents.get(n, ()):
            s |= self.children.get(p, set())
        return s - self.related(n)


tax = Taxonomy(df)
n_multi = sum(len(p) > 1 for p in tax.parents.values())
print(f"{len(tax.ids):,} nodes | {len(tax.categories):,} categories | "
      f"{n_multi:,} nodes with >1 parent")


181 nodes | 35 categories | 25 nodes with >1 parent


### Train / val / test split (by CVE, stratified by CWE)


In [6]:
def split_cves(df, val_frac=0.1, test_frac=0.1, seed=SEED):
    """
    Split by CVE-ID (a CVE with several CWE rows never straddles splits).
    Stratified on the CVE's primary CWE so rare CWEs show up in val/test too.
    CWEs with <3 CVEs stay in train only.
    If your CSV has a publish date, a time-based split is an even more honest
    test (CVE descriptions from the same advisory family are near-duplicates).
    """
    rng = random.Random(seed)
    cve_to_cwes = df.groupby("CVE-ID")["CWE-ID"].apply(set).to_dict()
    by_cwe = defaultdict(list)
    for cve, cwes in cve_to_cwes.items():
        by_cwe[min(cwes)].append(cve)
    train, val, test = [], [], []
    for _, cves in sorted(by_cwe.items()):
        cves = sorted(cves)
        rng.shuffle(cves)
        if len(cves) < 3:
            train += cves
            continue
        n_val = max(1, round(len(cves) * val_frac))
        n_test = max(1, round(len(cves) * test_frac))
        val += cves[:n_val]
        test += cves[n_val:n_val + n_test]
        train += cves[n_val + n_test:]
    return sorted(train), sorted(val), sorted(test)



In [7]:
train_cves, val_cves, test_cves = split_cves(df)
print(f"CVEs  train={len(train_cves):,}  val={len(val_cves):,}  test={len(test_cves):,}")


CVEs  train=48,066  val=6,013  test=6,013


### Retrieval + evaluation that mirrors your inference

`HierarchicalRetriever` is the deployment code path; the evaluator uses it, so
whatever you validate is what you ship.


In [8]:
from sentence_transformers.evaluation import SentenceEvaluator


def encode(model, texts, batch_size=128):
    return model.encode(
        list(texts), batch_size=batch_size, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )



class HierarchicalRetriever:
    """
    stage 1: score every category node, keep the top `n_categories`
    stage 2: BFS below those categories (up to `max_depth` levels, plus the
             category itself) and return the top-k by cosine similarity.
    """

    def __init__(self, model, tax, max_depth=2, include_category=True):
        self.model, self.tax = model, tax
        self.node_emb = encode(model, [tax.text[n] for n in tax.ids])
        self.cats = list(tax.categories)
        self.cat_idx = np.array([tax.idx[c] for c in self.cats], dtype=int)
        self.cand = {}
        for c in self.cats:
            members = tax.descendants(c, max_depth) + ([c] if include_category else [])
            self.cand[c] = np.array([tax.idx[m] for m in members], dtype=int)

    def search_scores(self, scores, k=3, n_categories=1, categories=None):
        """scores: cosine similarity of one query to every node (len = n_nodes)."""
        if categories is None:
            order = np.argsort(-scores[self.cat_idx])[:n_categories]
            categories = [self.cats[i] for i in order]
        pool = np.unique(np.concatenate([self.cand[c] for c in categories]))
        top = pool[np.argsort(-scores[pool])][:k]
        return list(categories), [self.tax.ids[i] for i in top]

    def search(self, texts, k=3, n_categories=1):
        S = encode(self.model, texts) @ self.node_emb.T
        return [self.search_scores(s, k, n_categories) for s in S]


class HierarchicalRetrievalEvaluator(SentenceEvaluator):
    def __init__(self, df, tax, cve_ids, name="hier", max_depth=2, k=3, n_geom_pairs=5000):
        super().__init__()
        self.tax, self.name, self.max_depth, self.k = tax, name, max_depth, k
        sub = df[df["CVE-ID"].isin(set(cve_ids))]
        text = sub.groupby("CVE-ID")["CVE-DESCRIPTION-NORMALIZED"].first()
        gold = sub.groupby("CVE-ID")["CWE-ID"].apply(set)
        ids = sorted(cve_ids)
        self.texts = [text[c] for c in ids]
        self.gold = [gold[c] for c in ids]
        self.primary_metric = f"{name}_e2e_hit{k}"

        # fixed sample of sibling / cross-category CWE pairs for the geometry metric
        rng = random.Random(0)
        sib = [(a, b) for a in tax.ids for b in tax.siblings(a) if a < b]
        self.sib_pairs = rng.sample(sib, min(n_geom_pairs, len(sib)))
        cross = []
        for _ in range(n_geom_pairs * 3):
            a, b = rng.choice(tax.ids), rng.choice(tax.ids)
            if a != b and b not in tax.related(a) and b not in tax.siblings(a):
                cross.append((a, b))
            if len(cross) >= n_geom_pairs:
                break
        self.cross_pairs = cross

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        tax, k = self.tax, self.k
        ret = HierarchicalRetriever(model, tax, self.max_depth)
        S = encode(model, self.texts) @ ret.node_emb.T

        m = defaultdict(float)
        n_all, n_cat = len(self.texts), 0
        for s, gold in zip(S, self.gold):
            # flat baseline: best-ranked gold CWE among ALL nodes
            rank = int((s > s[[tax.idx[g] for g in gold]].max()).sum()) + 1
            m["flat_hit1"] += rank <= 1
            m["flat_hit3"] += rank <= 3
            m["flat_mrr"] += 1.0 / rank

            gold_cats = set().union(*(tax.parents.get(g, set()) for g in gold))
            if not gold_cats:        # root-level CWE: no category to route through
                continue
            n_cat += 1
            # stage 1: is a true parent in the top-1 / top-3 categories?
            top_cats = [ret.cats[i] for i in np.argsort(-s[ret.cat_idx])[:3]]
            m["cat_hit1"] += top_cats[0] in gold_cats
            m["cat_hit3"] += any(c in gold_cats for c in top_cats)
            # stage 2 alone: told the right category (oracle), BFS below it
            _, top = ret.search_scores(s, k, categories=sorted(gold_cats))
            m["child_oracle_hit1"] += top[0] in gold
            m[f"child_oracle_hit{k}"] += any(t in gold for t in top)
            # full pipeline, 1 category and a 2-category beam
            _, top = ret.search_scores(s, k, n_categories=1)
            m["e2e_hit1"] += top[0] in gold
            m[f"e2e_hit{k}"] += any(t in gold for t in top)
            _, top = ret.search_scores(s, k, n_categories=2)
            m[f"e2e_beam2_hit{k}"] += any(t in gold for t in top)

        metrics = {}
        for key, v in m.items():
            metrics[key] = float(v / (n_all if key.startswith("flat") else max(n_cat, 1)))
        metrics["frac_with_category"] = n_cat / max(n_all, 1)

        # geometry (CWE-side, in-sample diagnostic): siblings should be closer than strangers
        E = ret.node_emb
        cos = lambda pairs: float(np.mean([E[tax.idx[a]] @ E[tax.idx[b]] for a, b in pairs])) if pairs else 0.0
        metrics["sibling_cos"] = cos(self.sib_pairs)
        metrics["cross_cat_cos"] = cos(self.cross_pairs)
        metrics["geom_gap"] = metrics["sibling_cos"] - metrics["cross_cat_cos"]

        metrics = self.prefix_name_to_metrics(metrics, self.name)
        self.store_metrics_in_model_card_data(model, metrics, epoch, steps)
        return metrics
def show(metrics, title):
    print(f"\n=== {title} ===")
    for k_, v in metrics.items():
        print(f"  {k_:38s} {v:.4f}")

/tmp/ipykernel_450/2827062844.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import SentenceEvaluator


### Training data

**Variant A (`A_hier_full`)** - three datasets, all MultipleNegativesRankingLoss:

| set | anchor -> positive | hard negatives | teaches |
|---|---|---|---|
| `cve_to_cwe` | CVE -> its CWE | a sibling CWE + a look-alike from another category | stage 2, and category separation |
| `cve_to_category` | CVE -> its parent CWE | nearest other category | stage 1 |
| `cwe_geometry` | CWE -> a sibling CWE | look-alike from another category | siblings closer / other categories apart |

**Variant B (`B_children_only`)** - one dataset, `(CVE, own CWE)` pairs, nothing else.
The only negatives are the other CWEs in the same batch, and `NO_DUPLICATES`
guarantees none of them is the same CWE as the positive.


In [9]:
def mine_hard_negatives(model, tax, top_m=10):
    """
    CWE-level hard-negative mining with the (still untrained) model.
      hard_node[n]: nearest nodes that are NOT related to n and NOT its siblings
                    (so: "looks similar, but a different category")
      hard_cat[n] : nearest *category* nodes that are not related to n
                    (siblings allowed: a neighbouring category is the hard case)
    """
    emb = encode(model, [tax.text[n] for n in tax.ids])
    sim = emb @ emb.T
    is_cat = np.array([n in tax.children for n in tax.ids])

    def top(s):
        order = np.argsort(-s)[:top_m]
        return [tax.ids[j] for j in order if np.isfinite(s[j])]

    hard_node, hard_cat = {}, {}
    for i, n in enumerate(tax.ids):
        s = sim[i].copy()
        s[[tax.idx[e] for e in tax.related(n)]] = -np.inf
        sc = s.copy()
        sc[~is_cat] = -np.inf
        hard_cat[n] = top(sc)
        s[[tax.idx[e] for e in tax.siblings(n)]] = -np.inf
        hard_node[n] = top(s)
    return hard_node, hard_cat


def build_training_sets(df, tax, cve_ids, hard_node, hard_cat, seed=SEED, geo_per_node=3):
    from datasets import Dataset
    rng = random.Random(seed)
    sub = df[df["CVE-ID"].isin(set(cve_ids))]
    text = sub.groupby("CVE-ID")["CVE-DESCRIPTION-NORMALIZED"].first()
    gold = sub.groupby("CVE-ID")["CWE-ID"].apply(set)

    child_rows, cat_rows, geo_rows = [], [], []
    for cve in sorted(cve_ids):
        golds = gold[cve]
        forbidden = set()                      # never use a true answer as a negative
        for g in golds:
            forbidden |= tax.related(g)

        for g in sorted(golds):
            # ---- stage 2: CVE -> own CWE vs [sibling, other-category look-alike]
            sibs = sorted(s for s in tax.siblings(g) if s not in forbidden)
            cross = [c for c in hard_node[g] if c not in forbidden][:5]
            rng.shuffle(cross)
            negs = ([rng.choice(sibs)] if sibs else []) + cross
            negs = list(dict.fromkeys(negs))[:2]
            if len(negs) == 2:
                child_rows.append({
                    "anchor": text[cve], "positive": tax.text[g],
                    "negative_1": tax.text[negs[0]], "negative_2": tax.text[negs[1]],
                })

            # ---- stage 1: CVE -> parent category vs nearest other category
            for p in sorted(tax.parents.get(g, ())):
                pool = [c for c in hard_cat[p] if c not in forbidden][:5]
                if pool:
                    cat_rows.append({
                        "anchor": text[cve], "positive": tax.text[p],
                        "negative": tax.text[rng.choice(pool)],
                    })

    # ---- CWE-level geometry (taxonomy structure, no CVEs involved)
    for n in tax.ids:
        sibs = sorted(tax.siblings(n))
        cross = hard_node[n][:5]
        if not sibs or not cross:
            continue
        for s in rng.sample(sibs, min(geo_per_node, len(sibs))):
            geo_rows.append({
                "anchor": tax.text[n], "positive": tax.text[s],
                "negative": tax.text[rng.choice(cross)],
            })

    sets = {
        "cve_to_cwe": Dataset.from_list(child_rows),
        "cve_to_category": Dataset.from_list(cat_rows),
        "cwe_geometry": Dataset.from_list(geo_rows),
    }
    return sets


def build_children_only_set(df, tax, cve_ids):
    """Variant B: (anchor=CVE, positive=its CWE). No explicit negatives of any kind."""
    from datasets import Dataset
    sub = df[df["CVE-ID"].isin(set(cve_ids))]
    text = sub.groupby("CVE-ID")["CVE-DESCRIPTION-NORMALIZED"].first()
    gold = sub.groupby("CVE-ID")["CWE-ID"].apply(set)
    rows = [
        {"anchor": text[c], "positive": tax.text[g]}
        for c in sorted(cve_ids) for g in sorted(gold[c])
    ]
    return Dataset.from_list(rows)

In [10]:
# Baseline + hard-negative mining use the untrained model (LoRA B=0 == base model).
base_model = load_model()

val_evaluator = HierarchicalRetrievalEvaluator(df, tax, val_cves, name="hier")
base_val = val_evaluator(base_model)
show(base_val, "BASELINE  validation")
test_evaluator = HierarchicalRetrievalEvaluator(df, tax, test_cves, name="test")
base_test = test_evaluator(base_model)          # stored for the final table, not used for decisions

hard_node, hard_cat = mine_hard_negatives(base_model, tax)

import gc
del base_model; gc.collect(); torch.cuda.empty_cache()

VARIANTS = {
    "A_hier_full":     build_training_sets(df, tax, train_cves, hard_node, hard_cat),
    "B_children_only": {"cve_to_cwe": build_children_only_set(df, tax, train_cves)},
}
for v, sets in VARIANTS.items():
    print(v, {k: len(d) for k, d in sets.items()})
print(f"Distinct CWE texts: {len(tax.ids):,} (NO_DUPLICATES needs every text in a batch to be unique)")

Unsloth: Not planning a device map; SentenceTransformer moves the assembled model onto a single device. Using `sequential`.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Using fast encoder path for bert (torch.compile + SDPA)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: unsloth/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unsloth: torch.compile will be applied automatically if max_steps > 3270

=== BASELINE  validation ===
  hier_flat_hit1                         0.1921
  hier_flat_hit3                         0.3003
  hier_flat_mrr                          0.2902
  hier_cat_hit1                          0.1532
  hier_cat_hit3                          0.2999
  hier_child_oracle_hit1                 0.4462
  hier_child_oracle_hit3                 0.6905
  hier_e2e_hit1                          0.0369
  hier_e2e_hit3                          0.0737
  hier_e2e_beam2_hit3                    0.0993
  hier_frac_with_category                1.0000
  hier_sibling_cos                       0.4637
  hier_cross_cat_cos                     0.2698
  hier_geom_gap                          0.1939
A_hier_full {'cve_to_cwe': 48686, 'cve_to_category': 50497, 'cwe_geometry': 404}
B_children_only {'cve_to_cwe': 48686}
Distinct CWE texts: 181 (NO_DUPLICATES needs every text in a batch to be unique)


### Train each variant from an identical fresh model, evaluate on val + test


In [11]:
from sentence_transformers import (
    SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses,
)
from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers
from unsloth import is_bf16_supported

COMMON = dict(epochs = 3, lr = 1e-4, batch_size = 64)   # identical for every variant


def train_variant(vname, train_sets):
    model = load_model()
    trainer = SentenceTransformerTrainer(
        model = model,
        train_dataset = train_sets,
        loss = {k: losses.MultipleNegativesRankingLoss(model) for k in train_sets},
        evaluator = HierarchicalRetrievalEvaluator(df, tax, val_cves, name="hier"),
        args = SentenceTransformerTrainingArguments(
            output_dir = f"output/{vname}",
            num_train_epochs = COMMON["epochs"],
            per_device_train_batch_size = COMMON["batch_size"],
            learning_rate = COMMON["lr"],
            warmup_ratio = 0.05,
            lr_scheduler_type = "linear",
            fp16 = not is_bf16_supported(),
            bf16 = is_bf16_supported(),
            logging_steps = 50,
            report_to = "none",
            seed = SEED,
            batch_sampler = BatchSamplers.NO_DUPLICATES,
            multi_dataset_batch_sampler = MultiDatasetBatchSamplers.PROPORTIONAL,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            save_total_limit = 2,
            load_best_model_at_end = True,     # if this errors with the Unsloth/PEFT wrapper,
            metric_for_best_model = "eval_hier_e2e_hit3",   # set False and pick the epoch by hand
            greater_is_better = True,
        ),
    )
    stats = trainer.train()
    val = HierarchicalRetrievalEvaluator(df, tax, val_cves, name="hier")(model)
    test = test_evaluator(model)                 # one look at test per variant, after training
    model.save_pretrained(f"saved_{vname}_lora")
    model.tokenizer.save_pretrained(f"saved_{vname}_lora")
    minutes = stats.metrics["train_runtime"] / 60
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return val, test, minutes


results = {}
for vname, sets in VARIANTS.items():
    print(f"\n########## {vname} ##########")
    results[vname] = train_variant(vname, sets)
    print(f"{vname}: {results[vname][2]:.1f} min")



########## A_hier_full ##########
Unsloth: Not planning a device map; SentenceTransformer moves the assembled model onto a single device. Using `sequential`.
Unsloth: Using fast encoder path for bert (torch.compile + SDPA)


/tmp/ipykernel_450/341023141.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_450/341023141.py:4: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: unsloth/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unsloth: torch.compile will be applied automatically if max_steps > 3270


...

Unsloth: torch.compile will be applied automatically if max_steps > 3270


[sentence_transformers.base.training_args|WARNING]The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Hier Flat Hit1,Hier Flat Hit3,Hier Flat Mrr,Hier Cat Hit1,Hier Cat Hit3,Hier Child Oracle Hit1,Hier Child Oracle Hit3,Hier E2e Hit1,Hier E2e Hit3,Hier E2e Beam2 Hit3,Hier Frac With Category,Hier Sibling Cos,Hier Cross Cat Cos,Hier Geom Gap
1,0.437189,No log,0.804091,0.887743,0.853118,0.299518,0.552470,0.862465,0.954099,0.244969,0.286047,0.437219,1.000000,0.462035,0.298120,0.163915
2,0.379670,No log,0.821886,0.890737,0.864222,0.326127,0.608182,0.876268,0.954765,0.275071,0.309496,0.503243,1.000000,0.380302,0.206422,0.173880
3,0.266652,No log,0.821054,0.894063,0.864814,0.372692,0.626642,0.878929,0.956594,0.323798,0.357891,0.533178,1.000000,0.328145,0.141586,0.186559


B_children_only: 33.1 min


### Side-by-side results


In [12]:
def strip(m):                                    # "hier_e2e_hit3" / "test_e2e_hit3" -> "e2e_hit3"
    return {k.split("_", 1)[1]: v for k, v in m.items()}

ORDER = ["cat_hit1", "cat_hit3", "child_oracle_hit1", "child_oracle_hit3",
         "e2e_hit1", "e2e_hit3", "e2e_beam2_hit3", "flat_hit1", "flat_hit3", "flat_mrr", "geom_gap"]

val_tbl = pd.DataFrame({"baseline": strip(base_val),
                        **{v: strip(r[0]) for v, r in results.items()}}).loc[ORDER]
test_tbl = pd.DataFrame({"baseline": strip(base_test),
                         **{v: strip(r[1]) for v, r in results.items()}}).loc[ORDER]

print("VALIDATION (use this to choose)\n", val_tbl.round(4).to_string())
print("\nTEST (report this, don't tune on it)\n", test_tbl.round(4).to_string())
val_tbl.to_csv("compare_val.csv"); test_tbl.to_csv("compare_test.csv")

winner = val_tbl.loc["e2e_hit3", list(results)].idxmax()
print(f"\nBest on validation e2e_hit3: {winner}  ->  saved_{winner}_lora/")


VALIDATION (use this to choose)
                    baseline  A_hier_full  B_children_only
cat_hit1             0.1532       0.8864           0.3727
cat_hit3             0.2999       0.9468           0.6266
child_oracle_hit1    0.4462       0.9247           0.8789
child_oracle_hit3    0.6905       0.9782           0.9566
e2e_hit1             0.0369       0.8468           0.3238
e2e_hit3             0.0737       0.8789           0.3579
e2e_beam2_hit3       0.0993       0.9119           0.5332
flat_hit1            0.1921       0.8683           0.8211
flat_hit3            0.3003       0.9288           0.8941
flat_mrr             0.2902       0.9035           0.8648
geom_gap             0.1939       0.2265           0.1866

TEST (report this, don't tune on it)
                    baseline  A_hier_full  B_children_only
cat_hit1             0.1515       0.8931           0.3685
cat_hit3             0.2935       0.9521           0.6295
child_oracle_hit1    0.4439       0.9375           0.8911


In [1]:
import glob, json, os, shutil
from transformers import AutoModel
from peft import PeftModel

def merge_lora(src, dst):
    # find the adapter wherever sentence-transformers put it (root or a subfolder)
    cfg_path = next(iter(sorted(glob.glob(f"{src}/**/adapter_config.json", recursive=True))), None)
    assert cfg_path, f"no adapter_config.json under {src}"
    tdir = os.path.dirname(cfg_path)
    base_id = json.load(open(cfg_path))["base_model_name_or_path"]

    base = AutoModel.from_pretrained(base_id).float()          # needs internet once
    merged = PeftModel.from_pretrained(base, tdir).merge_and_unload()

    # keep the whole sentence-transformers folder (modules.json, pooling, normalize,
    # tokenizer, max_seq_length), swap only the transformer weights
    shutil.copytree(src, dst, dirs_exist_ok=True)
    out = os.path.join(dst, os.path.relpath(tdir, src))
    for f in glob.glob(f"{out}/adapter_*"):
        os.remove(f)
    merged.save_pretrained(out)

for name in ["A_hier_full", "B_children_only"]:
    merge_lora(f"workspace/saved_{name}_lora", f"merged_{name}")

/home/regularpooria/Projects/CWE_CVE_Mapping/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1005 20:41:25.944000 1378958 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1005 20:41:26.007000 1378958 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6716.83it/s]
/home/regularpooria/Projects/CWE_CVE_Mapping/venv/lib/pyt